In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import seaborn as sns
import IPython.display as ipd

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap

import joblib
from sklearn import datasets
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.stats import multivariate_normal
from sklearn.covariance import LedoitWolf

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
file_path = ""
df = pd.read_csv(file_path)
df.head()

In [ ]:
# [11, 21, 18, 5, 23, 27]
CLUSTER_NUM = 27
df = df[df['cluster_id'] == CLUSTER_NUM].copy()

print(df.shape)

In [ ]:
# unlabeled data from Revised_final_diagnosis
df_unlabeled = df[df['Revised_final_diagnosis'].isnull()]
df_unlabeled.drop(columns=['Revised_final_diagnosis'], inplace=True)
print(df_unlabeled.shape)
print(df_unlabeled.columns)

In [ ]:
df_unlabeled.head()

In [ ]:
df_train_initial = pd.read_csv("")
df_test1_initial = pd.read_csv("")

print(df_train_initial.shape, df_test1_initial.shape)

In [ ]:
patient_ids_train = df_train_initial['pseudo_patient_id'].values
patient_ids_test1 = df_test1_initial['pseudo_patient_id'].values
print(patient_ids_train.shape, patient_ids_test1.shape)
# patient_ids_test2 = df_test2_initial['pseudo_patient_id'].values
# print(patient_ids_train.shape, patient_ids_test1.shape, patient_ids_test2.shape)

In [ ]:
# get the rows from the df which have those patient ids and correspondingly make the train and test sets

df_train = df[df["pseudo_patient_id"].isin(patient_ids_train)]
df_test1 = df[df["pseudo_patient_id"].isin(patient_ids_test1)]
# df_test2 = df[df["pseudo_patient_id"].isin(patient_ids_test2)]

print(df_train.shape)
print(df_test1.shape)
# print(df_test2.shape)

In [ ]:
df_train['Revised_final_diagnosis'].value_counts(dropna=False)

In [ ]:
df_test1['Revised_final_diagnosis'].value_counts(dropna=False)

In [ ]:
X_unlabeled = df_unlabeled[['z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8', 'z_9', 'z_10', 'z_11', 'z_12', 'z_13', 'z_14', 'z_15']].values

X_train = df_train[['z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8', 'z_9', 'z_10', 'z_11', 'z_12', 'z_13', 'z_14', 'z_15']].values

X_test1 = df_test1[['z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8', 'z_9', 'z_10', 'z_11', 'z_12', 'z_13', 'z_14', 'z_15']].values

print(X_unlabeled.shape, X_train.shape, X_test1.shape)

# X_test2 = df_test2[['z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8', 'z_9', 'z_10', 'z_11', 'z_12', 'z_13', 'z_14', 'z_15']].values

# print(X_unlabeled.shape, X_train.shape, X_test1.shape, X_test2.shape)

# KMEANS

In [ ]:
optimal_k = 15

kmeans = KMeans(n_clusters=optimal_k, random_state=42)
unlabeled_cluster_ids = kmeans.fit_predict(X_unlabeled)

# sil_score = silhouette_score(X_unlabeled, unlabeled_cluster_ids)
# print(f"Silhouette Score: {sil_score:.8f}")

In [ ]:
print(unlabeled_cluster_ids.shape)
print(unlabeled_cluster_ids)

In [ ]:
train_cluster_ids = kmeans.predict(X_train)
test1_cluster_ids = kmeans.predict(X_test1)
# test2_cluster_ids = kmeans.predict(X_test2)
print(train_cluster_ids.shape)
print(test1_cluster_ids.shape)
# print(test2_cluster_ids.shape)

In [ ]:
df_unlabeled['subcluster_id'] = unlabeled_cluster_ids
df_train['subcluster_id'] = train_cluster_ids
df_test1['subcluster_id'] = test1_cluster_ids
# df_test2['subcluster_id'] = test2_cluster_ids

In [ ]:
# df_unlabeled['cluster_id'] = unlabeled_cluster_ids
# df_train['cluster_id'] = train_cluster_ids
# df_test1['cluster_id'] = test1_cluster_ids
# # df_test2['cluster_id'] = test2_cluster_ids

In [ ]:
def fit_cluster_gaussians(X_unlabeled, cluster_ids):
    gaussians = {}

    for c in np.unique(cluster_ids):
        Xc = X_unlabeled[cluster_ids == c]

        if len(Xc) < 2:
            gaussians[c] = None
            continue

        mu = Xc.mean(axis=0)

        # Robust covariance estimation (important in high-D)
        cov = LedoitWolf().fit(Xc).covariance_

        gaussians[c] = multivariate_normal(mean=mu, cov=cov, allow_singular=True)

    return gaussians

def compute_gaussian_weights(X_labeled, cluster_ids, gaussians):
    n = len(X_labeled)
    log_weights = np.zeros(n)

    # Step 1: Compute logpdf for all points
    for i, (x, c) in enumerate(zip(X_labeled, cluster_ids)):
        g = gaussians.get(c)
        if g is None:
            log_weights[i] = -np.inf   # no gaussian → zero weight later
        else:
            log_weights[i] = g.logpdf(x)

    # Step 2: Normalize within clusters to get stable weights
    weights = np.zeros(n)

    unique_clusters = np.unique(cluster_ids)

    for c in unique_clusters:
        cluster_mask = (cluster_ids == c)
        cluster_logs = log_weights[cluster_mask]

        if np.all(np.isneginf(cluster_logs)):
            continue

        normalized_wgts = (cluster_logs - np.min(cluster_logs)) / (np.max(cluster_logs) - np.min(cluster_logs) + 1e-8)

        weights[cluster_mask] = normalized_wgts

    return weights

cluster_gaussians = fit_cluster_gaussians(X_unlabeled, unlabeled_cluster_ids)


In [ ]:
df_unlabeled['subcluster_weight'] = compute_gaussian_weights(X_unlabeled, unlabeled_cluster_ids, cluster_gaussians)
df_train['subcluster_weight'] = compute_gaussian_weights(X_train, train_cluster_ids, cluster_gaussians)
df_test1['subcluster_weight'] = compute_gaussian_weights(X_test1, test1_cluster_ids, cluster_gaussians)
# df_test2['subcluster_weight'] = compute_gaussian_weights(X_test2, test2_cluster_ids, cluster_gaussians)

In [ ]:
# df_unlabeled['cluster_weight'] = compute_gaussian_weights(X_unlabeled, unlabeled_cluster_ids, cluster_gaussians)
# df_train['cluster_weight'] = compute_gaussian_weights(X_train, train_cluster_ids, cluster_gaussians)
# df_test1['cluster_weight'] = compute_gaussian_weights(X_test1, test1_cluster_ids, cluster_gaussians)
# # df_test2['cluster_weight'] = compute_gaussian_weights(X_test2, test2_cluster_ids, cluster_gaussians)

In [ ]:
# df_unlabeled['subcluster_id'] = np.nan
# df_unlabeled['subcluster_weight'] = np.nan

# df_train['subcluster_id'] = np.nan
# df_train['subcluster_weight'] = np.nan

# df_test1['subcluster_id'] = np.nan
# df_test1['subcluster_weight'] = np.nan

# # df_test2['subcluster_id'] = np.nan
# # df_test2['subcluster_weight'] = np.nan

In [ ]:
df_unlabeled['Revised_final_diagnosis'] = np.nan

In [ ]:
print(df_unlabeled.shape)
print(df_train.shape)
print(df_test1.shape)
# print(df_test2.shape)

In [ ]:
df_train_filtered = df_train.copy()
df_test1_filtered = df_test1.copy()
# df_test2_filtered = df_test2.copy()

In [ ]:
# # keep samples only of disease of interest in df_train and df_test

# diseases_of_interest = ['Chronic lymphocytic leukemia', 'Acute lymphoblastic leukemia', 'Chronic myeloid leukemia', 'Acute promyelocytic leukemia', 'Acute leukemia', 'Multiple myeloma', 'Acute myeloid leukemia non M3', 'Aplastic anemia', 'Primary myelofibrosis', 'Eosinophilia']

# df_train_filtered = df_train[df_train['Revised_final_diagnosis'].isin(diseases_of_interest)]
# df_test1_filtered = df_test1[df_test1['Revised_final_diagnosis'].isin(diseases_of_interest)]
# df_test2_filtered = df_test2[df_test2['Revised_final_diagnosis'].isin(diseases_of_interest)]

In [ ]:
df_train_combined = pd.concat([df_unlabeled, df_train_filtered], ignore_index=True)
df_train_combined.shape

In [ ]:
# df_combined = pd.concat([df_unlabeled, df_train_filtered, df_test1_filtered, df_test2_filtered], ignore_index=True)
# df_combined.shape

In [ ]:
df_combined = pd.concat([df_unlabeled, df_train_filtered, df_test1_filtered], ignore_index=True)
df_combined.shape

In [ ]:
def weighted_cluster_purity_summary(df, cluster_col, label_col, weight_col):

    total_counts = (
        df.groupby(cluster_col)
          .size()
          .rename("total_points")
    )

    labeled_df = df[df[label_col].notna()]

    labeled_counts = (
        labeled_df.groupby(cluster_col)
                  .size()
                  .rename("labeled_points")
    )

    # Weighted class mass per cluster
    class_mass = (
        labeled_df
        .groupby([cluster_col, label_col])[weight_col]
        .sum()
        .rename(weight_col)
        .reset_index()
    )

    majority = (
        class_mass
        .sort_values(weight_col, ascending=False)
        .groupby(cluster_col)
        .first()
        .rename(columns={label_col: "majority_class",
                         weight_col: "majority_weight"})
    )

    total_weight = (
        labeled_df.groupby(cluster_col)[weight_col]
                  .sum()
                  .rename("total_weight")
    )

    purity = (
        (majority["majority_weight"] / total_weight)
        .rename("weighted_purity")
    )

    summary = (
        pd.concat([total_counts, labeled_counts, purity, majority["majority_class"]], axis=1)
          .fillna({"labeled_points": 0, "weighted_purity": np.nan})
          .reset_index()
          .rename(columns={cluster_col: "cluster_no"})
          .sort_values("cluster_no")
    )

    return summary


In [ ]:
weighted_purity_df = weighted_cluster_purity_summary(
    pd.concat([df_unlabeled, df_train_filtered], ignore_index=True),
    cluster_col="subcluster_id",
    label_col="Revised_final_diagnosis",
    weight_col="subcluster_weight"
)

In [ ]:
# weighted_purity_df = weighted_cluster_purity_summary(
#     pd.concat([df_unlabeled, df_train_filtered], ignore_index=True),
#     cluster_col="cluster_id",
#     label_col="Revised_final_diagnosis",
#     weight_col="cluster_weight"
# )

In [ ]:
#sort on total_points
weighted_purity_df.sort_values(by='total_points', ascending=False).reset_index(drop=True)

In [ ]:
# 11, 21, 18, 5, 23, 27 
cluster_id = 14
sample_data = df_train_combined[df_train_combined['subcluster_id']==cluster_id].copy()
sample_data = sample_data[sample_data['Revised_final_diagnosis'].notna()]

sample_data = sample_data.sort_values(['Revised_final_diagnosis'], ignore_index=False)

sample_data[['Revised_final_diagnosis', 'subcluster_weight']]

In [ ]:
df_combined.shape

In [ ]:
df_combined.head()

In [ ]:
cluster_to_labelandweight_dict = {
    row['cluster_no']: [
        row['labeled_points'], 
        row['weighted_purity'], 
        row['majority_class'] if pd.notna(row['weighted_purity']) else np.nan
    ] 
    for _, row in weighted_purity_df.iterrows()
}

print(cluster_to_labelandweight_dict)

In [ ]:
# cluster_to_labelandweight_dict[26] = [3, np.nan, 'Unknown']
# cluster_to_labelandweight_dict[15] = [3, np.nan, 'Unknown']
# cluster_to_labelandweight_dict[22] = [14, 0.64285714, 'Chronic lymphocytic leukemia']
# cluster_to_labelandweight_dict[2] = [3, np.nan, 'Unknown']
# cluster_to_labelandweight_dict[10] = [2, np.nan, 'Unknown']

# cluster_to_labelandweight_dict[3] = [5, np.nan, 'Unknown']

# cluster_to_labelandweight_dict[12] = [3, 0.66667, 'Acute promyelocytic leukemia']


print(cluster_to_labelandweight_dict)

In [ ]:
# leave_clusters = [11, 21, 18, 5, 23, 27]
leave_clusters = []

for index, row in df_combined.iterrows():
    cluster_id = row['subcluster_id']
    cluster_weight = row['subcluster_weight']

    if cluster_id in cluster_to_labelandweight_dict and cluster_id not in leave_clusters:
        labeled_points, weighted_purity, majority_class = cluster_to_labelandweight_dict[cluster_id]
        
        if pd.notna(weighted_purity):
            predicted_label = majority_class
            confidence = weighted_purity * cluster_weight
        else:
            predicted_label = "Unknown"
            confidence = np.nan
    else:
        predicted_label = np.nan
        confidence = np.nan

    df_combined.at[index, 'predicted_label'] = predicted_label
    df_combined.at[index, 'confidence'] = confidence

In [ ]:
df_combined.shape

In [ ]:
df_combined.head()

In [ ]:
df_combined['predicted_label'].value_counts(dropna=False)

In [ ]:
df_combined.shape

In [ ]:
original_df = pd.read_csv("")
print(original_df.shape)

In [ ]:
# combine df_combined into original df based on 'pseudo_patient_id' by replacing the rows of original_df with the rows of df_combined where 'pseudo_patient_id' matches and leave other rows untouched

original_df_indexed = original_df.set_index('pseudo_patient_id')
df_combined_indexed = df_combined.set_index('pseudo_patient_id')
original_df_indexed.update(df_combined_indexed)

original_df_indexed.reset_index(inplace=True)

original_df_indexed.shape

In [ ]:
original_df_indexed.head()

In [ ]:
# [11, 21, 18, 5, 23, 27]
CLUSTER_NUM = 27
sanity_check_df = original_df_indexed[original_df_indexed['cluster_id'] == CLUSTER_NUM].copy()

print(sanity_check_df.shape)

In [ ]:
sanity_check_df.head()

In [ ]:
sanity_check_df['predicted_label'].value_counts(dropna=False)

In [ ]:
# compare equality of sanity_check_df and df_combined based on the columns

compare_columns = ['pseudo_patient_id', 'z_0', 'z_1', 'z_2', 'z_3', 'z_4', 'z_5', 'z_6', 'z_7', 'z_8', 'z_9', 'z_10', 'z_11', 'z_12', 'z_13', 'z_14', 'z_15', 'Analyzer_model', 'Hospital', 'cluster_id', 'cluster_weight', 'subcluster_id', 'subcluster_weight', 'Revised_final_diagnosis', 'predicted_label', 'confidence']

# find number of matching rows and number of non-matching rows between sanity_check_df and df_combined based on the compare_columns

matching_rows = sanity_check_df[compare_columns].merge(df_combined[compare_columns], on=compare_columns, how='inner')
non_matching_rows = sanity_check_df[compare_columns].merge(df_combined[compare_columns], on=compare_columns, how='outer', indicator=True).query('_merge != "both"')

print(f"Number of rows in df_combined: {df_combined.shape}")
print(f"Number of matching rows: {matching_rows.shape}")
print(f"Number of non-matching rows: {non_matching_rows.shape}")